# Fine-tune BERT untuk Disaster Tweets

Notebook ini pasangan dari notebook TF-IDF. Datanya sama, pembersihan duplikatnya sama, validasinya sama (stratified 5-fold, skor F1), jadi skornya bisa dibandingkan langsung.

Yang berbeda: teks tidak diubah jadi TF-IDF, tapi dimasukkan ke **model BERT yang sudah pretrained**, lalu dilatih sebentar (*fine-tune*) di data kita.

**Alur:** data & duplikat → tokenizer & model → training per fold → evaluasi & threshold → (opsional) blend dengan TF-IDF + LR → submission.

**Kebutuhan:** `torch` versi CUDA, `transformers`, plus pandas, numpy, scikit-learn. GPU sangat disarankan. Di RTX 4060 (8GB) satu fold sekitar beberapa menit, tapi itu perkiraan kasar, lihat waktu aslinya di output.

In [1]:
import os, re, time, random, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_recall_curve
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"   # mixed precision: lebih cepat dan hemat VRAM
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if USE_AMP else "TIDAK ADA GPU")
if DEVICE == "cpu":
    print("PERINGATAN: berjalan di CPU. Sangat lambat untuk BERT base. Pasang torch versi CUDA dulu.")

C:\Users\SWIFT\PycharmProjects\JupyterProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cu130 | device: cuda | NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# # Kalau file CSV ada di folder lain, isi langsung, contoh: DATA_DIR = r"C:\Users\nama\Downloads\nlp-getting-started"
# DATA_DIR = os.environ.get("DATA_DIR", "")
#
# def find_data_dir():
#     if DATA_DIR and os.path.exists(os.path.join(DATA_DIR, "train.csv")):
#         return DATA_DIR
#     skip = {".git", ".venv", "venv", "env", "node_modules", "site-packages", "__pycache__"}
#     for root in ["/kaggle/input", "."]:
#         if not os.path.exists(root):
#             continue
#         base_depth = root.rstrip(os.sep).count(os.sep)
#         for dirpath, dirnames, files in os.walk(root):
#             dirnames[:] = [d for d in dirnames if d not in skip and not d.startswith(".")]
#             if dirpath.count(os.sep) - base_depth >= 3:
#                 dirnames[:] = []
#             if "train.csv" in files and "test.csv" in files:
#                 return dirpath
#     raise FileNotFoundError(
#         "train.csv dan test.csv tidak ditemukan.\n"
#         f"Folder kerja notebook saat ini: {os.getcwd()}\n"
#         "Taruh train.csv, test.csv, sample_submission.csv di folder ./data di samping notebook, atau isi DATA_DIR di atas."
#     )
#
# DATA_DIR = find_data_dir()
# train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
# test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
# sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
# print("data dari:", DATA_DIR, "| train:", train.shape, "| test:", test.shape)

train = pd.read_csv('data/movie_reviews.csv')
test = pd.read_csv('data/test_data.csv')

## 1. Data: duplikat dan cleaning

Duplikat dan label konflik ditangani **persis seperti notebook sebelumnya**, supaya skor CV BERT dan TF-IDF sebanding dan tidak ada teks yang sama di fold training dan validation.

**Preprocessing untuk BERT berbeda dari TF-IDF.** Jangan buang stopword, jangan stemming, dan tidak perlu huruf kecil manual (tokenizer yang mengurus). BERT memakai kata kecil dan urutan kata sebagai konteks, jadi membuangnya justru merugikan. Cleaning dibuat minimal: URL dan mention diganti token generik, entitas HTML dibuang, tanda `#` dibuang tapi katanya dipertahankan.

In [7]:
def norm_for_dup(s):
    return (s.fillna("").str.lower()
              .str.replace(r"http\S+", "", regex=True)
              .str.replace(r"\s+", " ", regex=True).str.strip())

train["norm"] = norm_for_dup(train["text"])
n_label = train.groupby("norm")["label"].nunique()
konflik = n_label[n_label > 1].index

train_clean = (train[~train["norm"].isin(konflik)]
               .drop_duplicates("norm")
               .reset_index(drop=True))
train_clean["text"] = train_clean["text"].fillna("")
print(f"baris awal: {len(train)} | grup konflik: {len(konflik)} | setelah dibersihkan: {len(train_clean)}")

def clean_text(s):
    s = re.sub(r"\d+", " ", s) #hapus angka
    s = re.sub(r"[^a-zA-Z0-9]", " ", s) #hapus non-alphabeth
    s = s.lower() #ubah semua menjadi lowercase
    s = re.sub(r"\s+", " ", s) #rapikan spasi
    words = [w for w in s.split() if len(w) > 2 and len(w) < 20] #filter kata pendek
    return " ".join(words) #gabungkan lagi jdi string dengan pemisah " "

X_text = train_clean["text"].astype(str).map(clean_text).tolist()
X_test = test["text"].fillna("").astype(str).map(clean_text).tolist()   # tidak ada baris test yang dibuang
y = train_clean["label"].values

baris awal: 40000 | grup konflik: 0 | setelah dibersihkan: 39740


## 2. Tokenizer dan pilihan model

**Tokenizer** BERT (WordPiece) memecah kata jadi potongan sub-kata yang ada di kosakatanya, sehingga kata asing atau typo tetap bisa diwakili. Hasilnya deretan ID angka, ditambah token khusus `[CLS]` di awal dan `[SEP]` di akhir. Padding dan *attention mask* dibuat per batch (dynamic padding), jadi batch tweet pendek tidak membuang komputasi.

**Pilihan model** (ubah `MODEL_NAME`):

| Model | Catatan |
|---|---|
| `bert-base-uncased` | BERT klasik, default yang aman |
| `roberta-base` | Sering sedikit lebih baik di teks sosial media, ukuran mirip |
| `distilbert-base-uncased` | Lebih kecil dan cepat, akurasi sedikit di bawah |
| `vinai/bertweet-base` | Dilatih khusus di tweet, tapi tokenizernya butuh paket tambahan (`emoji`) dan agak rewel |

Untuk teks bahasa Indonesia, ganti ke model bahasa Indonesia, misalnya `indobenchmark/indobert-base-p1`.

**Offline:** kalau folder `./models/<nama-model>` sudah ada, notebook memakainya dan tidak menyentuh internet. Cara menyimpannya ada di sel berikutnya.

In [8]:
MODEL_NAME = os.environ.get("MODEL_NAME", "bert-base-uncased")
LOCAL_DIR = os.path.join("models", MODEL_NAME.split("/")[-1])
MODEL_PATH = LOCAL_DIR if os.path.isdir(LOCAL_DIR) else MODEL_NAME
OFFLINE = os.path.isdir(MODEL_PATH)   # True kalau memuat dari folder lokal
print("model dimuat dari:", MODEL_PATH, "| mode lokal:", OFFLINE)

tok = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=OFFLINE)

MAX_LEN = 96
lens = np.array([len(x) for x in tok(X_text)["input_ids"]])
print("panjang token (persentil 50/90/95/99/max):", np.percentile(lens, [50, 90, 95, 99, 100]).astype(int))
print(f"tweet yang terpotong di MAX_LEN={MAX_LEN}: {(lens > MAX_LEN).mean():.2%}")

model dimuat dari: bert-base-uncased | mode lokal: False


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (574 > 512). Running this sequence through the model will result in indexing errors


panjang token (persentil 50/90/95/99/max): [ 153  404  529  801 2352]
tweet yang terpotong di MAX_LEN=96: 84.59%


**Simpan model untuk dipakai offline.** Jalankan sel ini **sekali saat internet menyala**, lalu salin folder `models/` ke flash drive sebagai cadangan. Setelah itu notebook otomatis memakai versi lokal.

Sel ini sengaja dikomentari supaya tidak jalan tanpa sengaja. Hapus tanda `#` untuk memakainya.

In [6]:
# JALANKAN SEKALI SAAT ONLINE (hapus tanda # di baris-baris berikut):
# _tok = AutoTokenizer.from_pretrained(MODEL_NAME)
# _mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# _tok.save_pretrained(LOCAL_DIR); _mdl.save_pretrained(LOCAL_DIR)
# print("tersimpan di:", LOCAL_DIR)

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 101.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

tersimpan di: models\bert-base-uncased


## 3. Pengaturan training

Training ditulis dengan loop PyTorch biasa (bukan `Trainer`), supaya setiap langkah kelihatan dan tidak bergantung pada perubahan API antar versi `transformers`.

| Pengaturan | Nilai | Alasan |
|---|---|---|
| `LR` | 2e-5 | Fine-tune BERT butuh learning rate kecil agar bobot pretrained tidak rusak |
| `EPOCHS` | 3 | Lebih dari itu di data sekecil ini mudah overfit |
| `BATCH_SIZE` | 32 | Turunkan ke 16 kalau VRAM habis (`CUDA out of memory`) |
| `WARMUP` | 10% | Learning rate naik pelan di awal, lalu turun linear |
| mixed precision | otomatis | `bfloat16` di GPU modern: lebih cepat, VRAM lebih hemat |

`RUN_FOLDS` menentukan berapa fold yang dijalankan. Set ke `1` untuk percobaan cepat; untuk hasil akhir pakai `5` (sama dengan `N_FOLDS`).

Pesan `Some weights ... newly initialized` itu **normal**: kepala klasifikasi di atas BERT memang baru dan dilatih dari nol.

In [9]:
N_FOLDS = 5
RUN_FOLDS = 5      # set 1 untuk uji cepat
EPOCHS = 3
BATCH_SIZE = 32
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01

def encode(texts):
    return tok(list(texts), truncation=True, max_length=MAX_LEN)["input_ids"]

def make_items(texts, labels=None):
    ids = encode(texts)
    return [{"ids": ids[i], "y": -1 if labels is None else int(labels[i])} for i in range(len(ids))]

def collate(batch):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["ids"])
        ids[i, :n] = torch.tensor(b["ids"])
        mask[i, :n] = 1
    return ids, mask, torch.tensor([b["y"] for b in batch])

items = make_items(X_text, y)
test_items = make_items(X_test)

@torch.no_grad()
def predict_proba(model, data):
    model.eval()
    out = []
    for ids, mask, _ in DataLoader(data, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate):
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
    return torch.cat(out).numpy()

def train_fold(tr_items, va_items, y_va):
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=2, local_files_only=OFFLINE).to(DEVICE)

    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=LR)
    loader = DataLoader(tr_items, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    total = EPOCHS * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

    for ep in range(EPOCHS):
        model.train()
        t0, run = time.time(), 0.0
        for ids, mask, yb in loader:
            ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(input_ids=ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        p = predict_proba(model, va_items)
        print(f"  epoch {ep + 1}/{EPOCHS} | loss {run / len(loader):.4f} | "
              f"val F1 {f1_score(y_va, (p >= 0.5).astype(int)):.4f} | {time.time() - t0:.0f} detik")
    return model, p   # prediksi validasi dari epoch terakhir (bukan epoch "terbaik", supaya tidak optimis)

## 4. Stratified CV dengan fold-ensemble

Setiap fold melatih satu model baru dari BERT pretrained (bobot awalnya selalu bersih, tidak terbawa dari fold sebelumnya). Dari tiap fold kita dapat dua hal:

- **Prediksi out-of-fold (OOF)** untuk bagian validation, dipakai menghitung skor yang jujur dan mencari threshold.
- **Prediksi untuk test**; rata-rata dari semua fold dipakai sebagai prediksi akhir. Merata-ratakan beberapa model biasanya sedikit lebih stabil daripada satu model, tanpa biaya tambahan di luar training fold itu sendiri.

Perhatikan **loss dan val F1 per epoch**: kalau loss training terus turun tapi val F1 naik lalu turun, itu tanda overfit. Kalau loss tidak turun sama sekali, cek learning rate dan pastikan device-nya `cuda`.

In [10]:
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)   # sama dengan notebook TF-IDF
oof = np.full(len(y), np.nan)
test_prob = np.zeros(len(test_items))
n_done = 0

for fold, (tr, va) in enumerate(cv.split(X_text, y)):
    if fold >= RUN_FOLDS:
        break
    print(f"\n=== Fold {fold + 1}/{N_FOLDS} ===")
    t0 = time.time()
    model, p_va = train_fold([items[i] for i in tr], [items[i] for i in va], y[va])
    oof[va] = p_va
    test_prob += predict_proba(model, test_items)
    n_done += 1
    print(f"fold selesai dalam {time.time() - t0:.0f} detik")
    del model
    if USE_AMP:
        torch.cuda.empty_cache()

test_prob /= n_done
mask = ~np.isnan(oof)
print(f"\nF1 OOF BERT @0.5 ({mask.sum()} baris): {f1_score(y[mask], (oof[mask] >= 0.5).astype(int)):.4f}")
os.makedirs("outputs", exist_ok=True)
np.save("outputs/bert_oof.npy", oof); np.save("outputs/bert_test.npy", test_prob)


=== Fold 1/5 ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1038.95it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

  epoch 1/3 | loss 0.3855 | val F1 0.8821 | 205 detik
  epoch 2/3 | loss 0.2240 | val F1 0.8818 | 201 detik
  epoch 3/3 | loss 0.1319 | val F1 0.8887 | 209 detik
fold selesai dalam 634 detik

=== Fold 2/5 ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 785.81it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

  epoch 1/3 | loss 0.3878 | val F1 0.8666 | 202 detik
  epoch 2/3 | loss 0.2203 | val F1 0.8742 | 283 detik
  epoch 3/3 | loss 0.1337 | val F1 0.8735 | 202 detik
fold selesai dalam 703 detik

=== Fold 3/5 ===


...

  epoch 1/3 | loss 0.3923 | val F1 0.8788 | 190 detik
  epoch 2/3 | loss 0.2252 | val F1 0.8928 | 232 detik
  epoch 3/3 | loss 0.1380 | val F1 0.8917 | 201 detik
fold selesai dalam 640 detik

=== Fold 4/5 ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 700.71it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

  epoch 1/3 | loss 0.3907 | val F1 0.8788 | 205 detik
  epoch 2/3 | loss 0.2274 | val F1 0.8813 | 204 detik
  epoch 3/3 | loss 0.1394 | val F1 0.8845 | 212 detik
fold selesai dalam 636 detik

=== Fold 5/5 ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 353.80it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

  epoch 1/3 | loss 0.3903 | val F1 0.8837 | 212 detik
  epoch 2/3 | loss 0.2264 | val F1 0.8873 | 217 detik
  epoch 3/3 | loss 0.1375 | val F1 0.8882 | 237 detik
fold selesai dalam 680 detik

F1 OOF BERT @0.5 (39740 baris): 0.8853


## 5. Evaluasi, threshold, dan error analysis

Threshold dicari dari prediksi OOF (bukan dari test) dan baru dipakai kalau kenaikan F1 lebih dari 0.005, sama seperti di notebook TF-IDF.

Baca juga prediksi yang salah. BERT biasanya memperbaiki kasus kiasan ("this song is fire"), jadi bandingkan contoh salahnya dengan yang salah di model TF-IDF: kalau polanya berbeda, itu alasan bagus untuk menggabungkan keduanya (bagian 6).

In [12]:
def tune_threshold(y_true, prob, default=0.5, margin=0.005):
    prec, rec, thr = precision_recall_curve(y_true, prob)
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    i = int(np.argmax(f1s[:-1]))
    f_def = f1_score(y_true, (prob >= default).astype(int))
    use = float(thr[i]) if (f1s[i] - f_def) > margin else default
    return use, f_def, float(f1s[i])

ym, pm = y[mask], oof[mask]
thr_bert, f_def, f_best = tune_threshold(ym, pm)
print(f"F1 @0.5: {f_def:.4f} | F1 terbaik: {f_best:.4f} | threshold dipakai: {thr_bert:.3f}")

pred = (pm >= thr_bert).astype(int)
print(classification_report(ym, pred, digits=3))
print(pd.DataFrame(confusion_matrix(ym, pred), index=["asli 0", "asli 1"], columns=["pred 0", "pred 1"]))

wrong = train_clean[mask].assign(prob=pm, pred=pred)
wrong = wrong[wrong["label"] != wrong["pred"]]
print(f"\n{len(wrong)} salah prediksi. Contoh:")
for _, r in wrong.sample(min(15, len(wrong)), random_state=SEED).iterrows():
    print(f"[asli={r['label']} p={r['prob']:.2f}] {r['text'][:140]}")

F1 @0.5: 0.8853 | F1 terbaik: 0.8864 | threshold dipakai: 0.500
              precision    recall  f1-score   support

           0      0.892     0.875     0.883     19862
           1      0.877     0.894     0.885     19878

    accuracy                          0.884     39740
   macro avg      0.884     0.884     0.884     39740
weighted avg      0.884     0.884     0.884     39740

        pred 0  pred 1
asli 0   17372    2490
asli 1    2112   17766

4602 salah prediksi. Contoh:
[asli=1 p=0.18] I just watched this movie for the first time after finishing the book last week Whats the problem here Folks admit that the performances are
[asli=0 p=0.83] Go to the video store and get the original I do not understand why Hollywood has that need to take a perfect foreign movie and remake it Mos
[asli=0 p=0.50] Without reiterating what was said above about this movie I would like to add that I was looking forward to watching this filmthe castlocatio
[asli=1 p=0.05] I make just one apology

## 6. Opsional: blend dengan TF-IDF + Logistic Regression

BERT dan TF-IDF melihat teks dengan cara berbeda, jadi kesalahannya sering tidak sama. Rata-rata berbobot dari dua probabilitas kadang menaikkan skor sedikit. Bobotnya dipilih dari prediksi OOF.

LR di sini memakai fold yang sama dan dilatih di dalam `Pipeline`, jadi tidak ada kebocoran. Kalau blend tidak lebih baik dari BERT sendirian (selisih di bawah sekitar 0.003 itu bisa cuma noise), pakai BERT saja.

In [16]:
def tfidf_lr():
    feats = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])
    return Pipeline([("features", feats), ("clf", LogisticRegression(max_iter=2000, C=3))])

lr_oof = cross_val_predict(tfidf_lr(), X_text, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
lr_test = tfidf_lr().fit(X_text, y).predict_proba(X_test)[:, 1]
print(f"F1 OOF LR @0.5: {f1_score(ym, (lr_oof[mask] >= 0.5).astype(int)):.4f}")

best_w, best_f = 1.0, -1
for w in np.linspace(0, 1, 11):
    f = f1_score(ym, ((w * pm + (1 - w) * lr_oof[mask]) >= 0.5).astype(int))
    print(f"  bobot BERT {w:.1f} -> F1 {f:.4f}")
    if f > best_f:
        best_w, best_f = float(w), f
print("bobot BERT terpilih:", best_w)

blend_oof = best_w * oof + (1 - best_w) * lr_oof
blend_test = best_w * test_prob + (1 - best_w) * lr_test
thr_blend, f_def_b, f_best_b = tune_threshold(ym, blend_oof[mask])
print(f"blend: F1 @0.5 {f_def_b:.4f} | threshold dipakai {thr_blend:.3f}")

KeyboardInterrupt: 

## 7. Submission

Dua file dibuat: `submission_bert.csv` dan `submission_blend.csv`. Bandingkan F1 OOF keduanya di output, lalu upload yang lebih baik. Kalau selisihnya kecil, upload keduanya (biasanya ada batas jumlah submission per hari) dan lihat mana yang lebih tinggi di leaderboard, tapi jangan menyetel ulang berdasarkan skor leaderboard.

Tidak ada baris test yang dibuang. Jumlah baris dan `id` dicek sama dengan `sample_submission.csv`.

In [ ]:
# Latih ulang pada seluruh data berlabel.
baseline.fit(X, y)

# Prediksi data test yang tidak memiliki label.
test_pred = baseline.predict(X_test)

submission = pd.DataFrame({
    "Id": df_test["Id"],
    "label": test_pred,
})

# Pemeriksaan format dasar.
assert submission.columns.tolist() == ["Id", "label"]
assert len(submission) == len(df_test)
assert submission["Id"].equals(df_test["Id"])
assert submission["label"].notna().all()
assert set(submission["label"].unique()).issubset(set(y.unique()))

submission = pd.DataFrame({"Id": df_test["Id"], "label": test_pred})
submission.to_csv("submission_sgd.csv", index=False)

print(submission["label"].value_counts(normalize=True).round(3))
submission.head()

In [15]:
def save_submission(prob, thr, path):
    sub = pd.DataFrame({"Id": test["Id"], "label": (prob >= thr).astype(int)})
    # assert len(sub) == len(sample_sub) and set(sub["Id"]) == set(sample_sub["Id"])
    sub.to_csv(path, index=False)
    return sub["label"].mean()

r1 = save_submission(test_prob, thr_bert, "submission_bert.csv")
# r2 = save_submission(blend_test, thr_blend, "submission_blend.csv")

print(f"submission_bert.csv  | F1 OOF {f1_score(ym, (pm >= thr_bert).astype(int)):.4f} | porsi analisis sentimen {r1:.3f}")
# print(f"submission_blend.csv | F1 OOF {f1_score(ym, (blend_oof[mask] >= thr_blend).astype(int)):.4f} | porsi analisis sentimen {r2:.3f}")
# print(f"porsi sentimen di data train (pembanding): {y.mean():.3f}")

submission_bert.csv  | F1 OOF 0.8853 | porsi analisis sentimen 0.511


## Catatan: kapan BERT cocok dan kapan tidak

**Cocok** kalau makna bergantung pada konteks (kiasan, negasi, sarkasme), datanya cukup (ribuan baris ke atas), ada GPU, dan bobot pretrained tersedia. Itu kasus Disaster Tweets.

**Kurang cocok** kalau datanya sangat kecil (ratusan baris, hasilnya labil antar seed), tidak ada GPU atau waktu terbatas, lingkungan tanpa internet dan bobot belum disiapkan, atau perlu menjelaskan fitur mana yang berpengaruh (TF-IDF + LR jauh lebih mudah ditafsirkan lewat koefisien).

**Kalau ada masalah:**

1. `CUDA out of memory`: turunkan `BATCH_SIZE` ke 16 dan/atau `MAX_LEN` ke 64.
2. Loss tidak turun, F1 tetap sekitar 0.5-0.6: cek `DEVICE`, coba `LR` 3e-5 atau 1e-5, pastikan labelnya benar.
3. Skor antar fold sangat berbeda: wajar di data kecil. Lihat rata-ratanya, bukan satu fold.
4. Mau coba model lain: ubah `MODEL_NAME` saja. Folder `outputs/` menyimpan probabilitas OOF/test untuk digabungkan nanti.
5. Skor leaderboard jauh di bawah F1 OOF: periksa lagi duplikat dan kebocoran di data.